In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/OZON')
os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
SUB_DIR = PROJECT_DIR / "submissions"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Рабочая директория: {PROJECT_DIR}")
print(f"Исходный train.parquet найден: {(PROJECT_DIR / 'train.parquet').exists()}")
print(f"Файлы в директории: {os.listdir(PROJECT_DIR)}")

Mounted at /content/drive
Рабочая директория: /content/drive/MyDrive/OZON
Исходный train.parquet найден: False
Файлы в директории: ['README.md', 'time_split.py', 'make_synthetic_data.py', 'features.py', 'data_loading.py', 'btyd_features.py', 'build_dataset.py', 'config.py', 'data', 'submissions', '__pycache__', 'uploads', 'models', 'train.py', 'predict.py']


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn


warnings.filterwarnings("ignore")


# ============================================================
# PATHS
# ============================================================

drive_path = Path(
    "/content/drive/MyDrive/OZON"
)

if drive_path.exists():
    os.chdir(drive_path)


DATA_DIR = Path("data")

OOF_DIR = (
    DATA_DIR /
    "oof"
)

SUB_DIR = Path(
    "submissions"
)

OOF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUB_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Используемое устройство: "
    f"{device}"
)


# ============================================================
# CONFIG
# ============================================================

SEEDS = [
    42,
    1337,
    2026,
]

BATCH_SIZE = 4096

EPOCHS = 80

LR = 1e-3

WEIGHT_DECAY = 1e-4

PATIENCE = 8

HIDDEN_DIM = 256

NUM_BLOCKS = 2

DROPOUT = 0.15

TOP_K_FEATURES = 100


# ============================================================
# MODEL
# ============================================================


class ResNetBlock(nn.Module):

    def __init__(
        self,
        dim,
        dropout=0.15,
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.BatchNorm1d(dim),

            nn.SiLU(),

            nn.Dropout(dropout),

            nn.Linear(
                dim,
                dim,
            ),

            nn.BatchNorm1d(dim),

            nn.SiLU(),

            nn.Dropout(dropout),

            nn.Linear(
                dim,
                dim,
            ),
        )

    def forward(self, x):

        return x + self.block(x)


class TabularResNet(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        num_blocks=2,
        dropout=0.15,
    ):

        super().__init__()

        self.input_layer = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            nn.BatchNorm1d(
                hidden_dim
            ),

            nn.SiLU(),
        )

        self.blocks = nn.ModuleList(

            [
                ResNetBlock(
                    hidden_dim,
                    dropout,
                )

                for _ in range(
                    num_blocks
                )
            ]
        )

        self.head = nn.Linear(
            hidden_dim,
            1,
        )

    def forward(self, x):

        x = self.input_layer(x)

        for block in self.blocks:

            x = block(x)

        return self.head(x)


# ============================================================
# METRIC
# ============================================================


def rmsle(
    y_true,
    y_pred,
):

    y_true = np.clip(
        np.asarray(y_true),
        0,
        None,
    )

    y_pred = np.clip(
        np.asarray(y_pred),
        0,
        None,
    )

    return float(
        np.sqrt(
            np.mean(
                (
                    np.log1p(y_pred)
                    -
                    np.log1p(y_true)
                )
                ** 2
            )
        )
    )


# ============================================================
# SEED
# ============================================================


def set_seed(
    seed: int,
):

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )

    # Для воспроизводимости.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# LOAD DATA
# ============================================================


def load_data():

    fold_files = sorted(
        DATA_DIR.glob(
            "fold_*.parquet"
        )
    )

    if not fold_files:

        raise FileNotFoundError(
            "Не найдены fold_*.parquet. "
            "Сначала запустите "
            "build_dataset.py."
        )

    folds = [
        pd.read_parquet(
            path
        )
        for path in fold_files
    ]

    test_path = (
        DATA_DIR /
        "test_features.parquet"
    )

    if not test_path.exists():

        raise FileNotFoundError(
            "Не найден test_features.parquet."
        )

    test_df = pd.read_parquet(
        test_path
    )

    print(
        f"Загружено CV-фолдов: "
        f"{len(folds)}"
    )

    for i, df in enumerate(folds):

        print(
            f"fold_{i}: "
            f"{len(df):,} строк"
        )

    print(
        f"test: "
        f"{len(test_df):,} строк"
    )

    return (
        folds,
        test_df,
    )


# ============================================================
# FEATURES
# ============================================================


def get_feature_columns(
    folds,
):

    exclude_cols = {
        "user_id",
        "target",
        "target_log",
        "fold",
        "cutoff_date",
        "first_order_dt",
        "last_order_dt",
    }

    all_features = [
        c
        for c in folds[0].columns
        if c not in exclude_cols
    ]

    # --------------------------------------------------------
    # Feature importance from LightGBM
    # --------------------------------------------------------

    fi_path = (
        DATA_DIR /
        "lgbm_feature_importances.csv"
    )

    if fi_path.exists():

        fi_df = pd.read_csv(
            fi_path
        )

        fi_df = fi_df.sort_values(
            "importance",
            ascending=False,
        )

        top_features = (
            fi_df[
                "feature"
            ]
            .tolist()
        )

        feature_cols = [
            c
            for c in top_features
            if c in all_features
        ]

        feature_cols = feature_cols[
            :TOP_K_FEATURES
        ]

        print(
            f"\nИспользуем Top-{len(feature_cols)} "
            f"признаков LightGBM."
        )

    else:

        print(
            "\nFeature importance не найден."
        )

        print(
            "Используем все признаки."
        )

        feature_cols = all_features

    print(
        f"Количество признаков: "
        f"{len(feature_cols)}"
    )

    return feature_cols


# ============================================================
# TENSOR CONVERSION
# ============================================================


def make_tensor(
    df,
    feature_cols,
):

    values = (
        df[
            feature_cols
        ]
        .fillna(0.0)
        .values
    )

    return torch.tensor(
        values,
        dtype=torch.float32,
        device=device,
    )


# ============================================================
# MAIN
# ============================================================


def main():

    folds, test_df = load_data()

    feature_cols = get_feature_columns(
        folds
    )

    # ========================================================
    # OOF ARRAYS
    # ========================================================

    # Здесь будут только validation folds 1..N.
    oof_predictions = []

    oof_targets = []

    oof_user_ids = []

    test_predictions_by_fold = []

    # ========================================================
    # TEMPORAL CV
    # ========================================================

    for val_fold_idx in range(
        1,
        len(folds),
    ):

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"TEMPORAL FOLD "
            f"{val_fold_idx}"
        )

        print(
            "=" * 70
        )

        # ----------------------------------------------------
        # TRAIN = ТОЛЬКО ПРОШЛЫЕ FOLDS
        # ----------------------------------------------------

        train_df = pd.concat(
            folds[
                :val_fold_idx
            ],
            ignore_index=True,
        )

        val_df = folds[
            val_fold_idx
        ]

        print(
            f"Train rows: "
            f"{len(train_df):,}"
        )

        print(
            f"Val rows: "
            f"{len(val_df):,}"
        )

        # ----------------------------------------------------
        # GPU TENSORS
        # ----------------------------------------------------

        X_train_raw = make_tensor(
            train_df,
            feature_cols,
        )

        y_train = torch.tensor(
            train_df[
                "target_log"
            ].values,
            dtype=torch.float32,
            device=device,
        ).unsqueeze(1)

        X_val_raw = make_tensor(
            val_df,
            feature_cols,
        )

        y_val = torch.tensor(
            val_df[
                "target_log"
            ].values,
            dtype=torch.float32,
            device=device,
        ).unsqueeze(1)

        X_test_raw = make_tensor(
            test_df,
            feature_cols,
        )

        # ----------------------------------------------------
        # NORMALIZATION
        # ----------------------------------------------------

        mean = X_train_raw.mean(
            dim=0,
            keepdim=True,
        )

        std = X_train_raw.std(
            dim=0,
            keepdim=True,
        )

        std = torch.clamp(
            std,
            min=1e-6,
        )

        X_train = (
            X_train_raw - mean
        ) / std

        X_val = (
            X_val_raw - mean
        ) / std

        X_test = (
            X_test_raw - mean
        ) / std

        n_train = len(
            X_train
        )

        n_val = len(
            X_val
        )

        # ====================================================
        # SEEDS
        # ====================================================

        seed_val_predictions = []

        seed_test_predictions = []

        for seed in SEEDS:

            print(
                f"\n--- Seed {seed} ---"
            )

            set_seed(
                seed
            )

            # ------------------------------------------------
            # MODEL
            # ------------------------------------------------

            model = TabularResNet(
                input_dim=len(
                    feature_cols
                ),
                hidden_dim=HIDDEN_DIM,
                num_blocks=NUM_BLOCKS,
                dropout=DROPOUT,
            ).to(device)

            # ------------------------------------------------
            # LOSS
            # ------------------------------------------------

            # RMSLE = RMSE(log1p(target), log1p(pred))
            #
            # Поэтому MSE на target_log
            # напрямую соответствует оптимизируемой
            # величине.

            criterion = nn.MSELoss()

            # ------------------------------------------------
            # OPTIMIZER
            # ------------------------------------------------

            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=LR,
                weight_decay=WEIGHT_DECAY,
            )

            scheduler = (
                torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer,
                    mode="min",
                    factor=0.5,
                    patience=2,
                )
            )

            # ------------------------------------------------
            # BEST CHECKPOINT
            # ------------------------------------------------

            best_loss = float(
                "inf"
            )

            best_state = None

            patience_counter = 0

            # =================================================
            # EPOCHS
            # =================================================

            for epoch in range(
                EPOCHS
            ):

                # ---------------------------------------------
                # TRAIN
                # ---------------------------------------------

                model.train()

                permutation = (
                    torch.randperm(
                        n_train,
                        device=device,
                    )
                )

                train_loss_sum = 0.0

                train_count = 0

                for start in range(
                    0,
                    n_train,
                    BATCH_SIZE,
                ):

                    idx = permutation[
                        start:
                        start + BATCH_SIZE
                    ]

                    bx = X_train[
                        idx
                    ]

                    by = y_train[
                        idx
                    ]

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    output = model(
                        bx
                    )

                    loss = criterion(
                        output,
                        by,
                    )

                    loss.backward()

                    # Небольшая защита
                    # от exploding gradients.
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=5.0,
                    )

                    optimizer.step()

                    batch_size = len(
                        bx
                    )

                    train_loss_sum += (
                        loss.item()
                        * batch_size
                    )

                    train_count += (
                        batch_size
                    )

                train_loss = (
                    train_loss_sum
                    /
                    max(
                        train_count,
                        1,
                    )
                )

                # ---------------------------------------------
                # VALIDATION
                # ---------------------------------------------

                model.eval()

                val_loss_sum = 0.0

                val_count = 0

                val_predictions = []

                with torch.no_grad():

                    for start in range(
                        0,
                        n_val,
                        BATCH_SIZE,
                    ):

                        bx = X_val[
                            start:
                            start + BATCH_SIZE
                        ]

                        by = y_val[
                            start:
                            start + BATCH_SIZE
                        ]

                        output = model(
                            bx
                        )

                        loss = criterion(
                            output,
                            by,
                        )

                        batch_size = len(
                            bx
                        )

                        val_loss_sum += (
                            loss.item()
                            * batch_size
                        )

                        val_count += (
                            batch_size
                        )

                        val_predictions.append(
                            output
                        )

                val_loss = (
                    val_loss_sum
                    /
                    max(
                        val_count,
                        1,
                    )
                )

                scheduler.step(
                    val_loss
                )

                # ---------------------------------------------
                # BEST MODEL
                # ---------------------------------------------

                if val_loss < best_loss:

                    best_loss = val_loss

                    best_state = {
                        key:
                        value.detach()
                        .cpu()
                        .clone()

                        for key, value
                        in model.state_dict()
                        .items()
                    }

                    patience_counter = 0

                else:

                    patience_counter += 1

                # ---------------------------------------------
                # LOG
                # ---------------------------------------------

                if (
                    epoch == 0
                    or
                    (epoch + 1) % 5 == 0
                ):

                    print(
                        f"Epoch "
                        f"{epoch + 1:03d} | "
                        f"train={train_loss:.6f} | "
                        f"val={val_loss:.6f} | "
                        f"best={best_loss:.6f} | "
                        f"lr={optimizer.param_groups[0]['lr']:.2e}"
                    )

                # ---------------------------------------------
                # EARLY STOPPING
                # ---------------------------------------------

                if (
                    patience_counter
                    >= PATIENCE
                ):

                    print(
                        f"Early stopping "
                        f"на epoch "
                        f"{epoch + 1}"
                    )

                    break

            # =================================================
            # RESTORE BEST MODEL
            # =================================================

            if best_state is None:

                raise RuntimeError(
                    "best_state == None"
                )

            model.load_state_dict(
                best_state
            )

            model.to(
                device
            )

            model.eval()

            # =================================================
            # BEST VALIDATION PREDICTION
            # =================================================

            val_preds = []

            with torch.no_grad():

                for start in range(
                    0,
                    n_val,
                    BATCH_SIZE,
                ):

                    output = model(
                        X_val[
                            start:
                            start + BATCH_SIZE
                        ]
                    )

                    val_preds.append(
                        output
                        .cpu()
                        .numpy()
                        .ravel()
                    )

            val_preds_log = np.concatenate(
                val_preds
            )

            # =================================================
            # BEST TEST PREDICTION
            # =================================================

            test_preds = []

            with torch.no_grad():

                for start in range(
                    0,
                    len(X_test),
                    BATCH_SIZE,
                ):

                    output = model(
                        X_test[
                            start:
                            start + BATCH_SIZE
                        ]
                    )

                    test_preds.append(
                        output
                        .cpu()
                        .numpy()
                        .ravel()
                    )

            test_preds_log = np.concatenate(
                test_preds
            )

            # ------------------------------------------------
            # SAVE
            # ------------------------------------------------

            seed_val_predictions.append(
                val_preds_log
            )

            seed_test_predictions.append(
                test_preds_log
            )

            print(
                f"Seed {seed}: "
                f"best log-MSE="
                f"{best_loss:.6f}"
            )

        # =====================================================
        # SEED ENSEMBLE
        # =====================================================

        avg_val_log = np.mean(
            seed_val_predictions,
            axis=0,
        )

        avg_test_log = np.mean(
            seed_test_predictions,
            axis=0,
        )

        # -----------------------------------------------------
        # LOG -> GMV
        # -----------------------------------------------------

        avg_val_gmv = np.expm1(
            np.clip(
                avg_val_log,
                0,
                None,
            )
        )

        avg_test_gmv = np.expm1(
            np.clip(
                avg_test_log,
                0,
                None,
            )
        )

        # -----------------------------------------------------
        # FOLD SCORE
        # -----------------------------------------------------

        fold_score = rmsle(
            val_df["target"].values,
            avg_val_gmv,
        )

        print(
            f"\nFOLD {val_fold_idx} "
            f"ENSEMBLE RMSLE = "
            f"{fold_score:.6f}"
        )

        # -----------------------------------------------------
        # OOF
        # -----------------------------------------------------

        oof_predictions.extend(
            avg_val_gmv
        )

        oof_targets.extend(
            val_df[
                "target"
            ].values
        )

        oof_user_ids.extend(
            val_df[
                "user_id"
            ].values
        )

        test_predictions_by_fold.append(
            avg_test_gmv
        )

        # -----------------------------------------------------
        # CLEANUP
        # -----------------------------------------------------

        del model

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

    # ========================================================
    # OVERALL OOF
    # ========================================================

    oof_predictions = np.asarray(
        oof_predictions
    )

    oof_targets = np.asarray(
        oof_targets
    )

    overall_score = rmsle(
        oof_targets,
        oof_predictions,
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "ИТОГОВЫЙ TEMPORAL "
        "OOF RMSLE:"
    )

    print(
        f"{overall_score:.6f}"
    )

    print(
        "=" * 70
    )

    # ========================================================
    # SAVE OOF
    # ========================================================

    oof_df = pd.DataFrame(
        {
            "user_id":
                oof_user_ids,

            "target":
                oof_targets,

            "pred":
                oof_predictions,
        }
    )

    oof_path = (
        OOF_DIR /
        "oof_nn.csv"
    )

    oof_df.to_csv(
        oof_path,
        index=False,
    )

    print(
        f"OOF сохранён: "
        f"{oof_path}"
    )

    # ========================================================
    # TEST PREDICTION
    # ========================================================

    # Усредняем predictions всех temporal folds.
    #
    # Последние folds обычно наиболее близки к test,
    # но пока оставляем простое среднее как baseline.

    final_test_preds = np.mean(
        test_predictions_by_fold,
        axis=0,
    )

    # ========================================================
    # SAVE SUBMISSION
    # ========================================================

    submission = pd.DataFrame(
        {
            "user_id":
                test_df[
                    "user_id"
                ].values,

            "predict":
                final_test_preds,
        }
    )

    submission_path = (
        SUB_DIR /
        "submission_nn.csv"
    )

    submission.to_csv(
        submission_path,
        index=False,
    )

    print(
        f"Submission сохранён: "
        f"{submission_path}"
    )

    print(
        "\nГотово."
    )


if __name__ == "__main__":

    main()

Используемое устройство: cuda
Загружено CV-фолдов: 6
fold_0: 250,000 строк
fold_1: 250,000 строк
fold_2: 250,000 строк
fold_3: 250,000 строк
fold_4: 250,000 строк
fold_5: 250,000 строк
test: 250,000 строк

Feature importance не найден.
Используем все признаки.
Количество признаков: 133

TEMPORAL FOLD 1
Train rows: 250,000
Val rows: 250,000

--- Seed 42 ---
Epoch 001 | train=3.493288 | val=2.976256 | best=2.976256 | lr=1.00e-03
Epoch 005 | train=2.909147 | val=2.971405 | best=2.938473 | lr=1.00e-03
Epoch 010 | train=2.880536 | val=2.925601 | best=2.925601 | lr=5.00e-04
Epoch 015 | train=2.868904 | val=2.928022 | best=2.925601 | lr=2.50e-04
Early stopping на epoch 18
Seed 42: best log-MSE=2.925601

--- Seed 1337 ---
Epoch 001 | train=3.415121 | val=3.004902 | best=3.004902 | lr=1.00e-03
Epoch 005 | train=2.912644 | val=2.953413 | best=2.948243 | lr=1.00e-03
Epoch 010 | train=2.883376 | val=2.941308 | best=2.934843 | lr=5.00e-04
Epoch 015 | train=2.875508 | val=2.931215 | best=2.925585 | 

In [ ]:
from pathlib import Path
import pandas as pd

SUB_DIR = Path('/content/drive/MyDrive/OZON/submissions')

lgb_path = SUB_DIR / 'submission_log_target.csv'
nn_path = SUB_DIR / 'submission_nn.csv'
cb_path = SUB_DIR / 'submission_catboost.csv'

lgb_sub = pd.read_csv(lgb_path) if lgb_path.exists() else None
nn_sub = pd.read_csv(nn_path) if nn_path.exists() else None
cb_sub = pd.read_csv(cb_path) if cb_path.exists() else None

for df in [lgb_sub, nn_sub, cb_sub]:
  if df is not None and 'target' in df.columns:
    df.rename(columns={'target': 'predict'}, inplace=True)

final_sub = lgb_sub.copy()

if cb_sub is not None:
  print('Создаём 3-компонентный бленд...')
  final_sub['predict'] = (
      0.50 * lgb_sub['predict']
      + 0.35 * cb_sub['predict']
      + 0.15 * nn_sub['predict']
  )
else:
  print('Создаём 2-компонентный бленд...')
  final_sub['predict'] = 0.85 * lgb_sub['predict'] + 0.15 * nn_sub['predict']

out_path = SUB_DIR / 'submission_blend_final.csv'
final_sub.to_csv(out_path, index=False)
print(f'Финальный сабмит сохранён в: {out_path}')

Создаём 3-компонентный бленд...
Финальный сабмит сохранён в: /content/drive/MyDrive/OZON/submissions/submission_blend_final.csv


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
SAMPLE_DIR = DATA_DIR / "samples"
SAMPLE_DIR.mkdir(exist_ok=True)

for f in sorted(DATA_DIR.glob("fold_*.parquet")):
    df = pd.read_parquet(f)
    sample = df.sample(
        n=min(10000, len(df)),
        random_state=42
    )
    sample.to_parquet(
        SAMPLE_DIR / f"{f.stem}_sample.parquet",
        index=False
    )

df = pd.read_parquet(DATA_DIR / "test_features.parquet")
df.sample(
    n=min(10000, len(df)),
    random_state=42
).to_parquet(
    SAMPLE_DIR / "test_sample.parquet",
    index=False
)

print("Готово:", list(SAMPLE_DIR.iterdir()))

Готово: [PosixPath('data/samples/fold_0_sample.parquet'), PosixPath('data/samples/fold_1_sample.parquet'), PosixPath('data/samples/fold_2_sample.parquet'), PosixPath('data/samples/fold_3_sample.parquet'), PosixPath('data/samples/fold_4_sample.parquet'), PosixPath('data/samples/fold_5_sample.parquet'), PosixPath('data/samples/test_sample.parquet')]
